# 🎓 ExamGuard — Complete Backend (Modules 1–5)
### Smart Online Exam System with AI Proctoring, Live Monitoring & PDF Reports

- **Module 1** — Authentication (JWT + bcrypt)
- **Module 2** — Exam Management
- **Module 3** — AI Proctoring (OpenCV)
- **Module 4** — Real-time Monitoring Dashboard
- **Module 5** — Result & Report Generation (PDF + CSV + Analytics)

---
Run cells in order. No account signup needed.


In [ ]:
# ── CELL 1: Install packages ────────────────────────────────────────────
!pip install flask flask-jwt-extended flask-cors bcrypt opencv-python-headless numpy fpdf2 -q
print('✅ All packages installed (Flask + OpenCV + fpdf2 for PDFs)')


In [ ]:
# ── CELL 2: Write app.py with ALL modules (1-5) ─────────────────────────

lines = [
    "import os, sqlite3, bcrypt, json, base64, time, io, csv",
    "import cv2, numpy as np",
    "from flask import Flask, request, jsonify, g, send_file",
    "from flask_jwt_extended import JWTManager, create_access_token, jwt_required, get_jwt_identity",
    "from flask_cors import CORS",
    "from datetime import timedelta, datetime",
    "from fpdf import FPDF",
    "",
    "app = Flask(__name__)",
    "app.config['JWT_SECRET_KEY'] = 'examguard-secret-change-in-prod-2024'",
    "app.config['JWT_ACCESS_TOKEN_EXPIRES'] = timedelta(hours=8)",
    "app.config['MAX_CONTENT_LENGTH'] = 5 * 1024 * 1024",
    "CORS(app, resources={r'/*': {'origins': '*'}})",
    "jwt = JWTManager(app)",
    "DB_PATH = 'examguard.db'",
    "",
    "FACE_CASCADE = cv2.CascadeClassifier(",
    "    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')",
    "print('✅ Face detection model loaded')",
    "",
    "LIVE_SESSIONS = {}",
    "",
    "def get_db():",
    "    import flask",
    "    if 'db' not in flask.g:",
    "        flask.g.db = sqlite3.connect(DB_PATH)",
    "        flask.g.db.row_factory = sqlite3.Row",
    "    return flask.g.db",
    "",
    "@app.teardown_appcontext",
    "def close_db(e=None):",
    "    import flask",
    "    db = flask.g.pop('db', None)",
    "    if db: db.close()",
    "",
    "def row_to_dict(row): return dict(row) if row else None",
    "def rows_to_list(rows): return [dict(r) for r in rows]",
    "",
    "def init_db():",
    "    with app.app_context():",
    "        db = sqlite3.connect(DB_PATH)",
    "        db.execute('''CREATE TABLE IF NOT EXISTS users (",
    "            user_id INTEGER PRIMARY KEY AUTOINCREMENT,",
    "            name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,",
    "            password TEXT NOT NULL, role TEXT NOT NULL DEFAULT \"student\",",
    "            created_at DATETIME DEFAULT CURRENT_TIMESTAMP)''')",
    "        db.execute('''CREATE TABLE IF NOT EXISTS exams (",
    "            exam_id INTEGER PRIMARY KEY AUTOINCREMENT,",
    "            title TEXT NOT NULL, description TEXT,",
    "            duration INTEGER NOT NULL, total_marks INTEGER NOT NULL,",
    "            created_by INTEGER, is_active INTEGER DEFAULT 1,",
    "            created_at DATETIME DEFAULT CURRENT_TIMESTAMP,",
    "            FOREIGN KEY (created_by) REFERENCES users(user_id))''')",
    "        db.execute('''CREATE TABLE IF NOT EXISTS questions (",
    "            question_id INTEGER PRIMARY KEY AUTOINCREMENT,",
    "            exam_id INTEGER NOT NULL, question_text TEXT NOT NULL,",
    "            option_a TEXT NOT NULL, option_b TEXT NOT NULL,",
    "            option_c TEXT NOT NULL, option_d TEXT NOT NULL,",
    "            correct_ans TEXT NOT NULL,",
    "            FOREIGN KEY (exam_id) REFERENCES exams(exam_id))''')",
    "        db.execute('''CREATE TABLE IF NOT EXISTS results (",
    "            result_id INTEGER PRIMARY KEY AUTOINCREMENT,",
    "            user_id INTEGER NOT NULL, exam_id INTEGER NOT NULL,",
    "            score INTEGER NOT NULL, total_marks INTEGER NOT NULL,",
    "            answers TEXT, submitted_at DATETIME DEFAULT CURRENT_TIMESTAMP,",
    "            FOREIGN KEY (user_id) REFERENCES users(user_id),",
    "            FOREIGN KEY (exam_id) REFERENCES exams(exam_id))''')",
    "        db.execute('''CREATE TABLE IF NOT EXISTS proctoring_logs (",
    "            log_id INTEGER PRIMARY KEY AUTOINCREMENT,",
    "            user_id INTEGER NOT NULL, exam_id INTEGER NOT NULL,",
    "            violation_type TEXT NOT NULL, screenshot TEXT,",
    "            face_count INTEGER, timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,",
    "            FOREIGN KEY (user_id) REFERENCES users(user_id),",
    "            FOREIGN KEY (exam_id) REFERENCES exams(exam_id))''')",
    "        db.commit()",
    "        db.close()",
    "        print('✅ Database ready')",
    "",
    "@app.route('/ping')",
    "def ping(): return jsonify(status='ok', message='ExamGuard backend running')",
    "",
    "# ════════════════════════════════════════════════════════",
    "# AUTH ROUTES (Module 1)",
    "# ════════════════════════════════════════════════════════",
    "@app.route('/auth/register', methods=['POST'])",
    "def register():",
    "    data = request.get_json()",
    "    for f in ['name','email','password','role']:",
    "        if not data.get(f): return jsonify(message=f'Missing: {f}'), 400",
    "    role = data['role'].lower()",
    "    if role not in ('student','teacher','admin'):",
    "        return jsonify(message='Invalid role'), 400",
    "    if len(data['password']) < 6:",
    "        return jsonify(message='Password min 6 chars'), 400",
    "    hashed = bcrypt.hashpw(data['password'].encode(), bcrypt.gensalt()).decode()",
    "    db = get_db()",
    "    try:",
    "        db.execute('INSERT INTO users(name,email,password,role) VALUES(?,?,?,?)',",
    "                   (data['name'].strip(), data['email'].strip().lower(), hashed, role))",
    "        db.commit()",
    "    except sqlite3.IntegrityError:",
    "        return jsonify(message='Email already registered'), 409",
    "    return jsonify(message='Account created'), 201",
    "",
    "@app.route('/auth/login', methods=['POST'])",
    "def login():",
    "    data = request.get_json()",
    "    email = (data.get('email') or '').strip().lower()",
    "    pw = data.get('password') or ''",
    "    if not email or not pw: return jsonify(message='Email and password required'), 400",
    "    db = get_db()",
    "    user = db.execute('SELECT * FROM users WHERE email=?', (email,)).fetchone()",
    "    if not user: return jsonify(message='No account found'), 401",
    "    if not bcrypt.checkpw(pw.encode(), user['password'].encode()):",
    "        return jsonify(message='Incorrect password'), 401",
    "    token = create_access_token(identity=str(user['user_id']))",
    "    return jsonify(access_token=token, user={",
    "        'user_id':user['user_id'],'name':user['name'],",
    "        'email':user['email'],'role':user['role']}), 200",
    "",
    "@app.route('/auth/me')",
    "@jwt_required()",
    "def me():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    user = db.execute('SELECT user_id,name,email,role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    return (jsonify(user=row_to_dict(user)),200) if user else (jsonify(message='Not found'),404)",
    "",
    "# ════════════════════════════════════════════════════════",
    "# EXAM ROUTES (Module 2)",
    "# ════════════════════════════════════════════════════════",
    "@app.route('/exam/create', methods=['POST'])",
    "@jwt_required()",
    "def create_exam():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    data = request.get_json()",
    "    for f in ['title','duration','total_marks']:",
    "        if not data.get(f): return jsonify(message=f'Missing: {f}'), 400",
    "    cur = db.execute(",
    "        'INSERT INTO exams(title,description,duration,total_marks,created_by) VALUES(?,?,?,?,?)',",
    "        (data['title'], data.get('description',''), int(data['duration']),",
    "         int(data['total_marks']), uid))",
    "    db.commit()",
    "    return jsonify(message='Exam created', exam_id=cur.lastrowid), 201",
    "",
    "@app.route('/exam/<int:exam_id>/question', methods=['POST'])",
    "@jwt_required()",
    "def add_question(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    data = request.get_json()",
    "    req = ['question_text','option_a','option_b','option_c','option_d','correct_ans']",
    "    for f in req:",
    "        if not data.get(f): return jsonify(message=f'Missing: {f}'), 400",
    "    if data['correct_ans'] not in ('a','b','c','d'):",
    "        return jsonify(message='correct_ans must be a, b, c, or d'), 400",
    "    db.execute(",
    "        'INSERT INTO questions(exam_id,question_text,option_a,option_b,option_c,option_d,correct_ans) VALUES(?,?,?,?,?,?,?)',",
    "        (exam_id, data['question_text'], data['option_a'], data['option_b'],",
    "         data['option_c'], data['option_d'], data['correct_ans']))",
    "    db.commit()",
    "    return jsonify(message='Question added'), 201",
    "",
    "@app.route('/exam/list')",
    "@jwt_required()",
    "def list_exams():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    sql = 'SELECT e.*, u.name as teacher_name, (SELECT COUNT(*) FROM questions WHERE exam_id=e.exam_id) as q_count FROM exams e JOIN users u ON e.created_by=u.user_id'",
    "    if caller['role'] in ('teacher','admin'):",
    "        exams = db.execute(sql+' WHERE e.created_by=? ORDER BY e.created_at DESC',(uid,)).fetchall()",
    "    else:",
    "        exams = db.execute(sql+' WHERE e.is_active=1 ORDER BY e.created_at DESC').fetchall()",
    "    return jsonify(exams=rows_to_list(exams)), 200",
    "",
    "@app.route('/exam/<int:exam_id>')",
    "@jwt_required()",
    "def get_exam(exam_id):",
    "    db = get_db()",
    "    exam = db.execute('SELECT * FROM exams WHERE exam_id=?',(exam_id,)).fetchone()",
    "    if not exam: return jsonify(message='Exam not found'), 404",
    "    questions = db.execute('SELECT question_id,question_text,option_a,option_b,option_c,option_d FROM questions WHERE exam_id=?',(exam_id,)).fetchall()",
    "    return jsonify(exam=row_to_dict(exam), questions=rows_to_list(questions)), 200",
    "",
    "@app.route('/exam/<int:exam_id>/submit', methods=['POST'])",
    "@jwt_required()",
    "def submit_exam(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    existing = db.execute('SELECT result_id FROM results WHERE user_id=? AND exam_id=?',(uid,exam_id)).fetchone()",
    "    if existing: return jsonify(message='Exam already submitted'), 409",
    "    data = request.get_json()",
    "    answers = data.get('answers', {})",
    "    questions = db.execute('SELECT question_id, correct_ans FROM questions WHERE exam_id=?',(exam_id,)).fetchall()",
    "    exam = db.execute('SELECT total_marks FROM exams WHERE exam_id=?',(exam_id,)).fetchone()",
    "    if not exam: return jsonify(message='Exam not found'), 404",
    "    if not questions: return jsonify(message='No questions'), 400",
    "    marks_per_q = exam['total_marks'] / len(questions)",
    "    score = 0",
    "    for q in questions:",
    "        qid = str(q['question_id'])",
    "        if answers.get(qid,'').lower() == q['correct_ans'].lower():",
    "            score += marks_per_q",
    "    score = round(score)",
    "    db.execute('INSERT INTO results(user_id,exam_id,score,total_marks,answers) VALUES(?,?,?,?,?)',",
    "        (uid, exam_id, score, exam['total_marks'], json.dumps(answers)))",
    "    db.commit()",
    "    LIVE_SESSIONS.pop((int(uid), exam_id), None)",
    "    passed = score >= (exam['total_marks'] * 0.5)",
    "    return jsonify(message='Submitted', score=score, total_marks=exam['total_marks'],",
    "        passed=passed, percentage=round((score/exam['total_marks'])*100,1)), 200",
    "",
    "@app.route('/exam/<int:exam_id>/results')",
    "@jwt_required()",
    "def get_all_results(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    results = db.execute('SELECT r.*, u.name, u.email FROM results r JOIN users u ON r.user_id=u.user_id WHERE r.exam_id=? ORDER BY r.score DESC',(exam_id,)).fetchall()",
    "    return jsonify(results=rows_to_list(results)), 200",
    "",
    "@app.route('/exam/<int:exam_id>/toggle', methods=['POST'])",
    "@jwt_required()",
    "def toggle_exam(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    db.execute('UPDATE exams SET is_active = CASE WHEN is_active=1 THEN 0 ELSE 1 END WHERE exam_id=?',(exam_id,))",
    "    db.commit()",
    "    exam = db.execute('SELECT is_active FROM exams WHERE exam_id=?',(exam_id,)).fetchone()",
    "    return jsonify(message='Updated', is_active=exam['is_active']), 200",
    "",
    "# ════════════════════════════════════════════════════════",
    "# AI PROCTORING (Module 3) + LIVE (Module 4)",
    "# ════════════════════════════════════════════════════════",
    "def detect_faces(image_b64):",
    "    try:",
    "        if ',' in image_b64:",
    "            image_b64 = image_b64.split(',', 1)[1]",
    "        img_bytes = base64.b64decode(image_b64)",
    "        nparr = np.frombuffer(img_bytes, np.uint8)",
    "        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)",
    "        if img is None: return -1, []",
    "        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)",
    "        faces = FACE_CASCADE.detectMultiScale(gray, 1.1, 5, minSize=(40,40))",
    "        return len(faces), [list(map(int, f)) for f in faces]",
    "    except Exception as e:",
    "        print(f'Face detection error: {e}')",
    "        return -1, []",
    "",
    "@app.route('/proctor/check', methods=['POST'])",
    "@jwt_required()",
    "def proctor_check():",
    "    uid = int(get_jwt_identity())",
    "    data = request.get_json()",
    "    exam_id = data.get('exam_id')",
    "    image_b64 = data.get('image')",
    "    if not exam_id or not image_b64:",
    "        return jsonify(message='Missing exam_id or image'), 400",
    "    face_count, faces = detect_faces(image_b64)",
    "    violation = None",
    "    if face_count == 0: violation = 'no_face'",
    "    elif face_count > 1: violation = 'multiple_faces'",
    "    key = (uid, exam_id)",
    "    now = time.time()",
    "    if key not in LIVE_SESSIONS:",
    "        LIVE_SESSIONS[key] = {'started_at': now, 'force_end': False, 'violations': 0}",
    "    session = LIVE_SESSIONS[key]",
    "    session['last_seen'] = now",
    "    session['face_count'] = face_count",
    "    session['last_snapshot'] = image_b64 if len(image_b64) < 100000 else None",
    "    if violation: session['violations'] = session.get('violations', 0) + 1",
    "    if violation:",
    "        db = get_db()",
    "        db.execute(",
    "            'INSERT INTO proctoring_logs(user_id,exam_id,violation_type,screenshot,face_count) VALUES(?,?,?,?,?)',",
    "            (uid, exam_id, violation, image_b64[:50000], face_count))",
    "        db.commit()",
    "    return jsonify(face_count=face_count, faces=faces, violation=violation,",
    "                   force_end=session.get('force_end', False)), 200",
    "",
    "@app.route('/proctor/log', methods=['POST'])",
    "@jwt_required()",
    "def log_violation():",
    "    uid = int(get_jwt_identity())",
    "    data = request.get_json()",
    "    exam_id = data.get('exam_id')",
    "    vtype = data.get('violation_type')",
    "    if not exam_id or not vtype:",
    "        return jsonify(message='Missing fields'), 400",
    "    db = get_db()",
    "    db.execute('INSERT INTO proctoring_logs(user_id,exam_id,violation_type) VALUES(?,?,?)',",
    "        (uid, exam_id, vtype))",
    "    db.commit()",
    "    key = (uid, exam_id)",
    "    if key in LIVE_SESSIONS:",
    "        LIVE_SESSIONS[key]['violations'] = LIVE_SESSIONS[key].get('violations', 0) + 1",
    "    return jsonify(message='Logged'), 201",
    "",
    "@app.route('/proctor/summary/<int:exam_id>')",
    "@jwt_required()",
    "def proctor_summary(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    summary = db.execute(",
    "        'SELECT u.user_id, u.name, u.email, COUNT(*) as total_violations, SUM(CASE WHEN p.violation_type=\"no_face\" THEN 1 ELSE 0 END) as no_face, SUM(CASE WHEN p.violation_type=\"multiple_faces\" THEN 1 ELSE 0 END) as multi_face, SUM(CASE WHEN p.violation_type=\"tab_switch\" THEN 1 ELSE 0 END) as tab_switch FROM proctoring_logs p JOIN users u ON p.user_id=u.user_id WHERE p.exam_id=? GROUP BY u.user_id ORDER BY total_violations DESC',",
    "        (exam_id,)).fetchall()",
    "    return jsonify(summary=rows_to_list(summary)), 200",
    "",
    "@app.route('/live/sessions')",
    "@jwt_required()",
    "def live_sessions():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    if caller['role'] == 'admin':",
    "        exam_ids = [r['exam_id'] for r in db.execute('SELECT exam_id FROM exams').fetchall()]",
    "    else:",
    "        exam_ids = [r['exam_id'] for r in db.execute('SELECT exam_id FROM exams WHERE created_by=?',(uid,)).fetchall()]",
    "    now = time.time()",
    "    STALE = 30",
    "    sessions_out = []",
    "    stale_keys = []",
    "    for key, sess in LIVE_SESSIONS.items():",
    "        student_id, exam_id = key",
    "        if exam_id not in exam_ids: continue",
    "        age = now - sess.get('last_seen', 0)",
    "        if age > 300:",
    "            stale_keys.append(key)",
    "            continue",
    "        u = db.execute('SELECT name, email FROM users WHERE user_id=?',(student_id,)).fetchone()",
    "        e = db.execute('SELECT title, duration FROM exams WHERE exam_id=?',(exam_id,)).fetchone()",
    "        if not u or not e: continue",
    "        elapsed = int(now - sess.get('started_at', now))",
    "        status = 'active' if age < STALE else 'idle'",
    "        sessions_out.append({",
    "            'student_id': student_id, 'student_name': u['name'],",
    "            'student_email': u['email'], 'exam_id': exam_id,",
    "            'exam_title': e['title'], 'exam_duration': e['duration'],",
    "            'elapsed_seconds': elapsed, 'last_seen_seconds_ago': int(age),",
    "            'face_count': sess.get('face_count'),",
    "            'violations': sess.get('violations', 0), 'status': status,",
    "            'has_snapshot': sess.get('last_snapshot') is not None })",
    "    for k in stale_keys: LIVE_SESSIONS.pop(k, None)",
    "    sessions_out.sort(key=lambda s: (-s['violations'], -s['elapsed_seconds']))",
    "    return jsonify(sessions=sessions_out, total=len(sessions_out)), 200",
    "",
    "@app.route('/live/snapshot/<int:exam_id>/<int:student_id>')",
    "@jwt_required()",
    "def live_snapshot(exam_id, student_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    key = (student_id, exam_id)",
    "    sess = LIVE_SESSIONS.get(key)",
    "    if not sess or not sess.get('last_snapshot'):",
    "        return jsonify(message='No live snapshot available'), 404",
    "    return jsonify(snapshot=sess['last_snapshot'], face_count=sess.get('face_count'),",
    "        last_seen_seconds_ago=int(time.time() - sess.get('last_seen', 0)),",
    "        violations=sess.get('violations', 0)), 200",
    "",
    "@app.route('/live/violations')",
    "@jwt_required()",
    "def live_violations():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    if caller['role'] == 'admin':",
    "        exam_filter = ''",
    "        params = ()",
    "    else:",
    "        exam_ids = [r['exam_id'] for r in db.execute('SELECT exam_id FROM exams WHERE created_by=?',(uid,)).fetchall()]",
    "        if not exam_ids: return jsonify(violations=[]), 200",
    "        placeholders = ','.join('?' for _ in exam_ids)",
    "        exam_filter = f' WHERE p.exam_id IN ({placeholders})'",
    "        params = tuple(exam_ids)",
    "    logs = db.execute(",
    "        f'SELECT p.log_id, p.exam_id, u.name as student_name, e.title as exam_title, p.violation_type, p.face_count, p.timestamp FROM proctoring_logs p JOIN users u ON p.user_id=u.user_id JOIN exams e ON p.exam_id=e.exam_id{exam_filter} ORDER BY p.timestamp DESC LIMIT 50',",
    "        params).fetchall()",
    "    return jsonify(violations=rows_to_list(logs)), 200",
    "",
    "@app.route('/live/terminate/<int:exam_id>/<int:student_id>', methods=['POST'])",
    "@jwt_required()",
    "def terminate_session(exam_id, student_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    key = (student_id, exam_id)",
    "    if key not in LIVE_SESSIONS:",
    "        return jsonify(message='Session not active'), 404",
    "    LIVE_SESSIONS[key]['force_end'] = True",
    "    db.execute('INSERT INTO proctoring_logs(user_id,exam_id,violation_type) VALUES(?,?,?)',",
    "        (student_id, exam_id, 'terminated_by_teacher'))",
    "    db.commit()",
    "    return jsonify(message='Session marked for termination'), 200",
    "",
    "@app.route('/live/stats')",
    "@jwt_required()",
    "def live_stats():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    if caller['role'] == 'admin':",
    "        exam_ids = [r['exam_id'] for r in db.execute('SELECT exam_id FROM exams').fetchall()]",
    "    else:",
    "        exam_ids = [r['exam_id'] for r in db.execute('SELECT exam_id FROM exams WHERE created_by=?',(uid,)).fetchall()]",
    "    now = time.time()",
    "    active_count = 0",
    "    total_violations_live = 0",
    "    exams_with_activity = set()",
    "    for key, sess in LIVE_SESSIONS.items():",
    "        _, ex_id = key",
    "        if ex_id not in exam_ids: continue",
    "        if now - sess.get('last_seen', 0) < 30:",
    "            active_count += 1",
    "            exams_with_activity.add(ex_id)",
    "            total_violations_live += sess.get('violations', 0)",
    "    return jsonify(active_students=active_count, active_exams=len(exams_with_activity),",
    "                   total_live_violations=total_violations_live), 200",
    "",
    "# ════════════════════════════════════════════════════════",
    "# MODULE 5: REPORTS & ANALYTICS",
    "# ════════════════════════════════════════════════════════",
    "",
    "def compute_exam_analytics(exam_id):",
    "    '''Return dict of analytics for one exam.'''",
    "    db = get_db()",
    "    exam = db.execute('SELECT * FROM exams WHERE exam_id=?',(exam_id,)).fetchone()",
    "    if not exam: return None",
    "    results = db.execute(",
    "        'SELECT r.*, u.name, u.email FROM results r JOIN users u ON r.user_id=u.user_id WHERE r.exam_id=? ORDER BY r.score DESC',",
    "        (exam_id,)).fetchall()",
    "    results = [dict(r) for r in results]",
    "    total_marks = exam['total_marks']",
    "    n = len(results)",
    "    if n == 0:",
    "        return {'exam': dict(exam), 'total_attempts': 0, 'results': [],",
    "                'avg_score': 0, 'pass_rate': 0, 'highest': 0, 'lowest': 0,",
    "                'pass_count': 0, 'fail_count': 0, 'score_distribution': {},",
    "                'question_stats': [], 'violation_stats': {}}",
    "    scores = [r['score'] for r in results]",
    "    passing = [s for s in scores if s >= total_marks * 0.5]",
    "    # Score distribution (buckets of 10%)",
    "    dist = {'0-20%':0,'20-40%':0,'40-60%':0,'60-80%':0,'80-100%':0}",
    "    for s in scores:",
    "        pct = (s / total_marks) * 100 if total_marks else 0",
    "        if pct < 20: dist['0-20%'] += 1",
    "        elif pct < 40: dist['20-40%'] += 1",
    "        elif pct < 60: dist['40-60%'] += 1",
    "        elif pct < 80: dist['60-80%'] += 1",
    "        else: dist['80-100%'] += 1",
    "    # Question-level analysis: how many got each question correct",
    "    questions = db.execute('SELECT * FROM questions WHERE exam_id=?',(exam_id,)).fetchall()",
    "    q_stats = []",
    "    for q in questions:",
    "        qid = str(q['question_id'])",
    "        correct_count = 0",
    "        for r in results:",
    "            try:",
    "                answers = json.loads(r['answers'] or '{}')",
    "                if answers.get(qid, '').lower() == q['correct_ans'].lower():",
    "                    correct_count += 1",
    "            except: pass",
    "        q_stats.append({",
    "            'question_id': q['question_id'],",
    "            'question_text': q['question_text'],",
    "            'correct_ans': q['correct_ans'],",
    "            'correct_count': correct_count,",
    "            'incorrect_count': n - correct_count,",
    "            'correct_pct': round(correct_count / n * 100, 1) if n else 0})",
    "    # Violation stats",
    "    viol_rows = db.execute(",
    "        'SELECT violation_type, COUNT(*) as cnt FROM proctoring_logs WHERE exam_id=? GROUP BY violation_type',",
    "        (exam_id,)).fetchall()",
    "    v_stats = {r['violation_type']: r['cnt'] for r in viol_rows}",
    "    return {",
    "        'exam': dict(exam),",
    "        'total_attempts': n,",
    "        'results': results,",
    "        'avg_score': round(sum(scores)/n, 1),",
    "        'avg_percentage': round(sum(scores)/n / total_marks * 100, 1) if total_marks else 0,",
    "        'pass_rate': round(len(passing)/n * 100, 1),",
    "        'highest': max(scores),",
    "        'lowest': min(scores),",
    "        'pass_count': len(passing),",
    "        'fail_count': n - len(passing),",
    "        'score_distribution': dist,",
    "        'question_stats': q_stats,",
    "        'violation_stats': v_stats,",
    "    }",
    "",
    "# ── Analytics endpoint (JSON for charts) ────────────────",
    "@app.route('/report/analytics/<int:exam_id>')",
    "@jwt_required()",
    "def exam_analytics(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    analytics = compute_exam_analytics(exam_id)",
    "    if not analytics: return jsonify(message='Exam not found'), 404",
    "    # Trim results (don't send answer JSON to save bandwidth)",
    "    for r in analytics['results']: r.pop('answers', None)",
    "    return jsonify(analytics), 200",
    "",
    "# ── PDF: Individual student report ──────────────────────",
    "class ReportPDF(FPDF):",
    "    def header(self):",
    "        self.set_fill_color(79, 142, 247)",
    "        self.rect(0, 0, 210, 30, 'F')",
    "        self.set_font('Helvetica', 'B', 20)",
    "        self.set_text_color(255, 255, 255)",
    "        self.set_xy(10, 8)",
    "        self.cell(0, 8, 'ExamGuard', ln=1)",
    "        self.set_font('Helvetica', '', 10)",
    "        self.cell(0, 5, 'Smart Online Examination System', ln=1)",
    "        self.set_text_color(0, 0, 0)",
    "        self.ln(15)",
    "    def footer(self):",
    "        self.set_y(-15)",
    "        self.set_font('Helvetica', 'I', 8)",
    "        self.set_text_color(128, 128, 128)",
    "        self.cell(0, 10, f'Page {self.page_no()}  |  Generated {datetime.now().strftime(\"%Y-%m-%d %H:%M\")}', align='C')",
    "",
    "def safe_text(s):",
    "    '''Convert Unicode strings to latin-1-safe for fpdf.'''",
    "    if s is None: return ''",
    "    return str(s).encode('latin-1', 'replace').decode('latin-1')",
    "",
    "@app.route('/report/student/<int:exam_id>/<int:student_id>.pdf')",
    "@jwt_required()",
    "def student_report_pdf(exam_id, student_id):",
    "    uid = int(get_jwt_identity())",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    # Students can only see their own, teachers can see anyone",
    "    if caller['role'] == 'student' and uid != student_id:",
    "        return jsonify(message='Cannot view other students\\' reports'), 403",
    "    student = db.execute('SELECT * FROM users WHERE user_id=?',(student_id,)).fetchone()",
    "    exam = db.execute('SELECT * FROM exams WHERE exam_id=?',(exam_id,)).fetchone()",
    "    result = db.execute('SELECT * FROM results WHERE user_id=? AND exam_id=?',(student_id, exam_id)).fetchone()",
    "    if not student or not exam:",
    "        return jsonify(message='Not found'), 404",
    "    violations = db.execute(",
    "        'SELECT violation_type, timestamp FROM proctoring_logs WHERE user_id=? AND exam_id=? ORDER BY timestamp',",
    "        (student_id, exam_id)).fetchall()",
    "    pdf = ReportPDF()",
    "    pdf.add_page()",
    "    # Title",
    "    pdf.set_font('Helvetica', 'B', 16)",
    "    pdf.cell(0, 10, 'Individual Student Report', ln=1)",
    "    pdf.ln(2)",
    "    # Student info card",
    "    pdf.set_fill_color(245, 246, 250)",
    "    pdf.set_font('Helvetica', 'B', 11)",
    "    pdf.cell(45, 8, 'Student Name:', fill=True)",
    "    pdf.set_font('Helvetica', '', 11)",
    "    pdf.cell(0, 8, safe_text(student['name']), ln=1, fill=True)",
    "    pdf.set_font('Helvetica', 'B', 11)",
    "    pdf.cell(45, 8, 'Email:', fill=True)",
    "    pdf.set_font('Helvetica', '', 11)",
    "    pdf.cell(0, 8, safe_text(student['email']), ln=1, fill=True)",
    "    pdf.set_font('Helvetica', 'B', 11)",
    "    pdf.cell(45, 8, 'Exam:', fill=True)",
    "    pdf.set_font('Helvetica', '', 11)",
    "    pdf.cell(0, 8, safe_text(exam['title']), ln=1, fill=True)",
    "    pdf.set_font('Helvetica', 'B', 11)",
    "    pdf.cell(45, 8, 'Duration:', fill=True)",
    "    pdf.set_font('Helvetica', '', 11)",
    "    pdf.cell(0, 8, f\"{exam['duration']} minutes\", ln=1, fill=True)",
    "    pdf.ln(8)",
    "    # Score",
    "    pdf.set_font('Helvetica', 'B', 14)",
    "    pdf.cell(0, 10, 'Result', ln=1)",
    "    pdf.ln(2)",
    "    if result:",
    "        pct = round(result['score'] / result['total_marks'] * 100, 1) if result['total_marks'] else 0",
    "        passed = pct >= 50",
    "        # Big score box",
    "        pdf.set_fill_color(52, 211, 153) if passed else pdf.set_fill_color(248, 113, 113)",
    "        pdf.set_text_color(255, 255, 255)",
    "        pdf.rect(10, pdf.get_y(), 60, 30, 'F')",
    "        pdf.set_xy(10, pdf.get_y())",
    "        pdf.set_font('Helvetica', 'B', 24)",
    "        pdf.cell(60, 20, f'{pct}%', align='C', ln=0)",
    "        pdf.set_xy(10, pdf.get_y()+20)",
    "        pdf.set_font('Helvetica', 'B', 10)",
    "        pdf.cell(60, 10, 'PASSED' if passed else 'FAILED', align='C')",
    "        pdf.set_text_color(0, 0, 0)",
    "        # Details next to it",
    "        pdf.set_xy(75, pdf.get_y()-30)",
    "        pdf.set_font('Helvetica', 'B', 11)",
    "        pdf.cell(0, 8, f'Score: {result[\"score\"]} / {result[\"total_marks\"]}', ln=1)",
    "        pdf.set_xy(75, pdf.get_y())",
    "        pdf.set_font('Helvetica', '', 10)",
    "        pdf.cell(0, 6, f'Submitted: {result[\"submitted_at\"]}', ln=1)",
    "        pdf.set_xy(75, pdf.get_y())",
    "        pdf.cell(0, 6, f'Percentage: {pct}%', ln=1)",
    "        pdf.set_xy(75, pdf.get_y())",
    "        pdf.cell(0, 6, f'Status: {\"Pass\" if passed else \"Fail\"} (threshold: 50%)', ln=1)",
    "        pdf.ln(15)",
    "    else:",
    "        pdf.set_font('Helvetica', 'I', 11)",
    "        pdf.set_text_color(128, 128, 128)",
    "        pdf.cell(0, 8, 'This student has not attempted the exam yet.', ln=1)",
    "        pdf.set_text_color(0, 0, 0)",
    "        pdf.ln(8)",
    "    # Proctoring section",
    "    pdf.set_font('Helvetica', 'B', 14)",
    "    pdf.cell(0, 10, 'Proctoring Report', ln=1)",
    "    pdf.ln(2)",
    "    if violations:",
    "        counts = {'no_face':0,'multiple_faces':0,'tab_switch':0,'terminated_by_teacher':0}",
    "        for v in violations:",
    "            counts[v['violation_type']] = counts.get(v['violation_type'], 0) + 1",
    "        # Summary boxes",
    "        w = 45",
    "        pdf.set_font('Helvetica', 'B', 10)",
    "        pdf.set_fill_color(255, 242, 220)",
    "        pdf.cell(w, 8, 'No Face', border=1, align='C', fill=True)",
    "        pdf.set_fill_color(254, 232, 232)",
    "        pdf.cell(w, 8, 'Multiple Faces', border=1, align='C', fill=True)",
    "        pdf.set_fill_color(220, 234, 255)",
    "        pdf.cell(w, 8, 'Tab Switch', border=1, align='C', fill=True)",
    "        pdf.set_fill_color(240, 240, 240)",
    "        pdf.cell(w, 8, 'Total', border=1, align='C', fill=True, ln=1)",
    "        pdf.set_font('Helvetica', 'B', 14)",
    "        pdf.cell(w, 10, str(counts['no_face']), border=1, align='C')",
    "        pdf.cell(w, 10, str(counts['multiple_faces']), border=1, align='C')",
    "        pdf.cell(w, 10, str(counts['tab_switch']), border=1, align='C')",
    "        pdf.cell(w, 10, str(len(violations)), border=1, align='C', ln=1)",
    "        pdf.ln(5)",
    "        # Risk level",
    "        total = len(violations)",
    "        risk = 'HIGH' if total >= 5 else 'MEDIUM' if total >= 2 else 'LOW'",
    "        risk_color = (248,113,113) if risk == 'HIGH' else (251,191,36) if risk == 'MEDIUM' else (52,211,153)",
    "        pdf.set_fill_color(*risk_color)",
    "        pdf.set_text_color(255, 255, 255)",
    "        pdf.set_font('Helvetica', 'B', 11)",
    "        pdf.cell(0, 10, f'Risk Level: {risk}', align='C', fill=True, ln=1)",
    "        pdf.set_text_color(0, 0, 0)",
    "        pdf.ln(4)",
    "        # Timeline",
    "        pdf.set_font('Helvetica', 'B', 11)",
    "        pdf.cell(0, 8, 'Violation Timeline:', ln=1)",
    "        pdf.set_font('Helvetica', '', 9)",
    "        for v in violations[:20]:",
    "            v_type = v['violation_type'].replace('_', ' ').title()",
    "            pdf.cell(0, 5, safe_text(f\"  - {v['timestamp']}  |  {v_type}\"), ln=1)",
    "        if len(violations) > 20:",
    "            pdf.set_font('Helvetica', 'I', 9)",
    "            pdf.cell(0, 5, f'  ... and {len(violations)-20} more', ln=1)",
    "    else:",
    "        pdf.set_fill_color(232, 245, 233)",
    "        pdf.set_text_color(52, 122, 90)",
    "        pdf.set_font('Helvetica', 'B', 11)",
    "        pdf.cell(0, 12, '  Clean exam - No violations recorded', align='L', fill=True, ln=1)",
    "        pdf.set_text_color(0, 0, 0)",
    "    # Return PDF as file",
    "    pdf_bytes = pdf.output(dest='S')",
    "    if isinstance(pdf_bytes, str): pdf_bytes = pdf_bytes.encode('latin-1')",
    "    return send_file(io.BytesIO(bytes(pdf_bytes)), mimetype='application/pdf',",
    "        as_attachment=True, download_name=f'report_{student[\"name\"].replace(\" \",\"_\")}_{exam[\"title\"].replace(\" \",\"_\")}.pdf')",
    "",
    "# ── PDF: Full class report ──────────────────────────────",
    "@app.route('/report/exam/<int:exam_id>.pdf')",
    "@jwt_required()",
    "def exam_report_pdf(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    analytics = compute_exam_analytics(exam_id)",
    "    if not analytics: return jsonify(message='Exam not found'), 404",
    "    exam = analytics['exam']",
    "    pdf = ReportPDF()",
    "    pdf.add_page()",
    "    pdf.set_font('Helvetica', 'B', 16)",
    "    pdf.cell(0, 10, 'Full Class Report', ln=1)",
    "    pdf.set_font('Helvetica', 'B', 13)",
    "    pdf.cell(0, 8, safe_text(exam['title']), ln=1)",
    "    if exam.get('description'):",
    "        pdf.set_font('Helvetica', 'I', 10)",
    "        pdf.cell(0, 6, safe_text(exam['description']), ln=1)",
    "    pdf.ln(4)",
    "    # Overview grid",
    "    pdf.set_font('Helvetica', 'B', 13)",
    "    pdf.cell(0, 8, 'Overview', ln=1)",
    "    pdf.ln(2)",
    "    # 4 boxes",
    "    w = 45; h = 22",
    "    def stat_box(label, value, color):",
    "        pdf.set_fill_color(*color)",
    "        pdf.rect(pdf.get_x(), pdf.get_y(), w, h, 'F')",
    "        x0, y0 = pdf.get_x(), pdf.get_y()",
    "        pdf.set_text_color(255,255,255)",
    "        pdf.set_xy(x0, y0+3)",
    "        pdf.set_font('Helvetica', 'B', 16)",
    "        pdf.cell(w, 10, str(value), align='C')",
    "        pdf.set_xy(x0, y0+13)",
    "        pdf.set_font('Helvetica', '', 8)",
    "        pdf.cell(w, 6, label, align='C')",
    "        pdf.set_text_color(0,0,0)",
    "        pdf.set_xy(x0+w+2, y0)",
    "    y_start = pdf.get_y()",
    "    stat_box('Attempts', analytics['total_attempts'], (79,142,247))",
    "    stat_box('Avg Score', f'{analytics[\"avg_percentage\"]}%', (52,211,153))",
    "    stat_box('Pass Rate', f'{analytics[\"pass_rate\"]}%', (245,158,11))",
    "    stat_box('Highest', analytics['highest'], (139,92,246))",
    "    pdf.set_y(y_start + h + 6)",
    "    # Score distribution as text bars",
    "    pdf.set_font('Helvetica', 'B', 12)",
    "    pdf.cell(0, 8, 'Score Distribution', ln=1)",
    "    pdf.ln(1)",
    "    max_bar = max(analytics['score_distribution'].values()) if analytics['score_distribution'] else 1",
    "    if max_bar == 0: max_bar = 1",
    "    for bucket, count in analytics['score_distribution'].items():",
    "        pdf.set_font('Helvetica', '', 9)",
    "        pdf.cell(25, 6, bucket)",
    "        # Bar",
    "        bar_w = (count / max_bar) * 100",
    "        pdf.set_fill_color(79, 142, 247)",
    "        pdf.rect(pdf.get_x(), pdf.get_y()+1, bar_w, 4, 'F')",
    "        pdf.set_x(pdf.get_x() + 105)",
    "        pdf.cell(0, 6, f'  {count}', ln=1)",
    "    pdf.ln(4)",
    "    # Results table",
    "    pdf.set_font('Helvetica', 'B', 12)",
    "    pdf.cell(0, 8, 'Results Table', ln=1)",
    "    pdf.set_fill_color(240, 240, 240)",
    "    pdf.set_font('Helvetica', 'B', 9)",
    "    pdf.cell(10, 7, '#', border=1, align='C', fill=True)",
    "    pdf.cell(60, 7, 'Student', border=1, fill=True)",
    "    pdf.cell(60, 7, 'Email', border=1, fill=True)",
    "    pdf.cell(25, 7, 'Score', border=1, align='C', fill=True)",
    "    pdf.cell(20, 7, 'Percent', border=1, align='C', fill=True)",
    "    pdf.cell(15, 7, 'Status', border=1, align='C', fill=True, ln=1)",
    "    pdf.set_font('Helvetica', '', 9)",
    "    for i, r in enumerate(analytics['results'], 1):",
    "        pct = round(r['score'] / r['total_marks'] * 100, 1) if r['total_marks'] else 0",
    "        passed = pct >= 50",
    "        pdf.cell(10, 6, str(i), border=1, align='C')",
    "        pdf.cell(60, 6, safe_text(r['name'])[:35], border=1)",
    "        pdf.cell(60, 6, safe_text(r['email'])[:35], border=1)",
    "        pdf.cell(25, 6, f\"{r['score']}/{r['total_marks']}\", border=1, align='C')",
    "        pdf.cell(20, 6, f'{pct}%', border=1, align='C')",
    "        pdf.set_text_color(52,122,90) if passed else pdf.set_text_color(180,50,50)",
    "        pdf.cell(15, 6, 'Pass' if passed else 'Fail', border=1, align='C', ln=1)",
    "        pdf.set_text_color(0,0,0)",
    "        if pdf.get_y() > 265:",
    "            pdf.add_page()",
    "    # New page: question analysis",
    "    if analytics['question_stats']:",
    "        pdf.add_page()",
    "        pdf.set_font('Helvetica', 'B', 14)",
    "        pdf.cell(0, 10, 'Question Analysis', ln=1)",
    "        pdf.set_font('Helvetica', '', 10)",
    "        pdf.cell(0, 6, 'Percentage of students who got each question correct', ln=1)",
    "        pdf.ln(2)",
    "        for i, q in enumerate(analytics['question_stats'], 1):",
    "            pdf.set_font('Helvetica', 'B', 10)",
    "            qtext = safe_text(q['question_text'])[:100]",
    "            pdf.multi_cell(0, 6, f'Q{i}. {qtext}')",
    "            pdf.set_font('Helvetica', '', 9)",
    "            # Bar showing correct %",
    "            bar_w = q['correct_pct']",
    "            color = (52,211,153) if bar_w >= 70 else (251,191,36) if bar_w >= 40 else (248,113,113)",
    "            pdf.set_fill_color(*color)",
    "            pdf.rect(pdf.get_x(), pdf.get_y()+1, bar_w, 5, 'F')",
    "            pdf.set_fill_color(240,240,240)",
    "            pdf.rect(pdf.get_x() + bar_w, pdf.get_y()+1, 100-bar_w, 5, 'F')",
    "            pdf.set_x(pdf.get_x() + 105)",
    "            pdf.cell(0, 6, f\"  {q['correct_pct']}% correct  ({q['correct_count']}/{q['correct_count']+q['incorrect_count']})\", ln=1)",
    "            pdf.ln(3)",
    "            if pdf.get_y() > 260: pdf.add_page()",
    "    pdf_bytes = pdf.output(dest='S')",
    "    if isinstance(pdf_bytes, str): pdf_bytes = pdf_bytes.encode('latin-1')",
    "    fname = f\"exam_report_{exam['title'].replace(' ','_')}.pdf\"",
    "    return send_file(io.BytesIO(bytes(pdf_bytes)), mimetype='application/pdf',",
    "        as_attachment=True, download_name=safe_text(fname))",
    "",
    "# ── CSV: Full class results ─────────────────────────────",
    "@app.route('/report/exam/<int:exam_id>.csv')",
    "@jwt_required()",
    "def exam_report_csv(exam_id):",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    caller = db.execute('SELECT role FROM users WHERE user_id=?',(uid,)).fetchone()",
    "    if not caller or caller['role'] not in ('teacher','admin'):",
    "        return jsonify(message='Teacher access required'), 403",
    "    analytics = compute_exam_analytics(exam_id)",
    "    if not analytics: return jsonify(message='Exam not found'), 404",
    "    # Fetch violation counts per student",
    "    v_counts = {}",
    "    v_rows = db.execute(",
    "        'SELECT user_id, COUNT(*) as cnt FROM proctoring_logs WHERE exam_id=? GROUP BY user_id',",
    "        (exam_id,)).fetchall()",
    "    for r in v_rows: v_counts[r['user_id']] = r['cnt']",
    "    out = io.StringIO()",
    "    w = csv.writer(out)",
    "    w.writerow(['Rank','Student Name','Email','Score','Total Marks','Percentage','Status','Violations','Submitted At'])",
    "    for i, r in enumerate(analytics['results'], 1):",
    "        pct = round(r['score'] / r['total_marks'] * 100, 1) if r['total_marks'] else 0",
    "        w.writerow([i, r['name'], r['email'], r['score'], r['total_marks'],",
    "                    f'{pct}%', 'Pass' if pct >= 50 else 'Fail',",
    "                    v_counts.get(r['user_id'], 0), r['submitted_at']])",
    "    csv_bytes = out.getvalue().encode('utf-8')",
    "    fname = f\"exam_results_{analytics['exam']['title'].replace(' ','_')}.csv\"",
    "    return send_file(io.BytesIO(csv_bytes), mimetype='text/csv',",
    "        as_attachment=True, download_name=safe_text(fname))",
    "",
    "# ── Student: get list of my results ─────────────────────",
    "@app.route('/my/results')",
    "@jwt_required()",
    "def my_results():",
    "    uid = get_jwt_identity()",
    "    db = get_db()",
    "    results = db.execute(",
    "        'SELECT r.*, e.title as exam_title, e.duration FROM results r JOIN exams e ON r.exam_id=e.exam_id WHERE r.user_id=? ORDER BY r.submitted_at DESC',",
    "        (uid,)).fetchall()",
    "    return jsonify(results=rows_to_list(results)), 200",
    "",
    "if __name__ == '__main__':",
    "    init_db()",
    "    app.run(port=5000, debug=False, use_reloader=False)",
]

with open('app.py', 'w') as f:
    f.write('\n'.join(lines))

print('✅ app.py written successfully')

import py_compile
try:
    py_compile.compile('app.py', doraise=True)
    print('✅ Syntax check passed')
except py_compile.PyCompileError as e:
    print(f'❌ Syntax error: {e}')


In [ ]:
# ── CELL 3: Start Flask server ──────────────────────────────────────────
import threading, subprocess, time, requests

def run_flask():
    proc = subprocess.Popen(
        ['python', 'app.py'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print('[Flask]', line, end='')

threading.Thread(target=run_flask, daemon=True).start()

print('⏳ Starting Flask (loading OpenCV takes ~15s)...')
for i in range(25):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:5000/ping', timeout=3)
        if r.status_code == 200:
            print(f'\n✅ Flask running! ({(i+1)*2}s)')
            break
    except:
        print(f'   Waiting... {(i+1)*2}s')
else:
    print('\n❌ Server did not start')


In [ ]:
# ── CELL 4: Public URL via Cloudflare Tunnel ────────────────────────────
import subprocess, threading, time, re

cf_url = None

def run_cf():
    global cf_url
    subprocess.run(['wget','-q','-nc',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O','cloudflared'], check=True)
    subprocess.run(['chmod','+x','cloudflared'], check=True)
    proc = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:5000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m: cf_url = m.group(0)

threading.Thread(target=run_cf, daemon=True).start()

print('⏳ Starting Cloudflare tunnel...')
for i in range(20):
    time.sleep(2)
    if cf_url: break
    print(f'   Waiting... {(i+1)*2}s')

if cf_url:
    print()
    print('='*60)
    print(f'  🌐  Public URL : {cf_url}')
    print(f'  📋  Paste into login.html when prompted')
    print('='*60)
    print()
    print('New Module 5 endpoints:')
    print(f'  GET  {cf_url}/report/analytics/<exam_id>          (JSON for charts)')
    print(f'  GET  {cf_url}/report/exam/<exam_id>.pdf           (full class PDF)')
    print(f'  GET  {cf_url}/report/exam/<exam_id>.csv           (CSV export)')
    print(f'  GET  {cf_url}/report/student/<exam_id>/<uid>.pdf  (student PDF)')
    print(f'  GET  {cf_url}/my/results                          (student\'s own results)')
else:
    print('❌ Tunnel failed')


In [ ]:
# ── CELL 5: Seed test accounts + sample exam + fake submissions ─────────
import requests, json

BASE = 'http://localhost:5000'
tokens = {}

accounts = [
    {'name':'Teacher Ali',  'email':'teacher@test.com', 'password':'teach123', 'role':'teacher'},
    {'name':'Student Alina','email':'alina@test.com',   'password':'stud123',  'role':'student'},
    {'name':'Student Bilal','email':'bilal@test.com',   'password':'stud123',  'role':'student'},
    {'name':'Student Cara', 'email':'cara@test.com',    'password':'stud123',  'role':'student'},
    {'name':'Student Danish','email':'danish@test.com', 'password':'stud123',  'role':'student'},
]
print('Creating accounts...')
for acc in accounts:
    r = requests.post(f'{BASE}/auth/register', json=acc)
    print(f'  {acc["role"]} ({acc["email"]}): {"✅" if r.status_code==201 else "exists"}')

for acc in accounts:
    r = requests.post(f'{BASE}/auth/login', json={'email':acc['email'],'password':acc['password']})
    tokens[acc['email']] = r.json().get('access_token')

th = {'Authorization':f'Bearer {tokens["teacher@test.com"]}','Content-Type':'application/json'}

print()
print('Creating sample exam...')
r = requests.post(f'{BASE}/exam/create', headers=th, json={
    'title':'Python Basics Quiz','description':'5 fundamental questions',
    'duration':10,'total_marks':10
})
exam_id = r.json().get('exam_id')
print(f'  Exam ID: {exam_id}')

questions = [
    {'question_text':'Which keyword defines a function?','option_a':'func','option_b':'def','option_c':'define','option_d':'fun','correct_ans':'b'},
    {'question_text':'What does len("hello") return?','option_a':'4','option_b':'6','option_c':'5','option_d':'0','correct_ans':'c'},
    {'question_text':'Which type is immutable?','option_a':'list','option_b':'dict','option_c':'set','option_d':'tuple','correct_ans':'d'},
    {'question_text':'Symbol for single-line comment?','option_a':'//','option_b':'#','option_c':'--','option_d':'/*','correct_ans':'b'},
    {'question_text':'Output of type(3.14)?','option_a':'int','option_b':'str','option_c':'float','option_d':'double','correct_ans':'c'},
]
for i, q in enumerate(questions, 1):
    requests.post(f'{BASE}/exam/{exam_id}/question', headers=th, json=q)

# Fake submissions with varying scores - question IDs 1-5 based on order
print()
print('Simulating student submissions...')
# 1=b, 2=c, 3=d, 4=b, 5=c are correct answers
submissions = {
    'alina@test.com': {'1':'b','2':'c','3':'d','4':'b','5':'c'},  # 5/5 → 100%
    'bilal@test.com': {'1':'b','2':'c','3':'d','4':'b','5':'a'},  # 4/5 → 80%
    'cara@test.com':  {'1':'b','2':'a','3':'d','4':'b','5':'c'},  # 4/5 → 80%
    'danish@test.com':{'1':'a','2':'a','3':'a','4':'a','5':'a'},  # 0/5 → 0%
}
for email, answers in submissions.items():
    sh = {'Authorization':f'Bearer {tokens[email]}','Content-Type':'application/json'}
    r = requests.post(f'{BASE}/exam/{exam_id}/submit', headers=sh, json={'answers':answers})
    d = r.json()
    print(f'  {email}: {d.get("score")}/{d.get("total_marks")} ({d.get("percentage")}%) - {"Pass" if d.get("passed") else "Fail"}')

# Fake proctoring violations for realistic report
print()
print('Simulating proctoring violations...')
violations = [
    ('bilal@test.com', 'no_face'),
    ('bilal@test.com', 'tab_switch'),
    ('danish@test.com', 'no_face'),
    ('danish@test.com', 'no_face'),
    ('danish@test.com', 'multiple_faces'),
    ('danish@test.com', 'tab_switch'),
    ('danish@test.com', 'tab_switch'),
]
for email, vtype in violations:
    sh = {'Authorization':f'Bearer {tokens[email]}','Content-Type':'application/json'}
    requests.post(f'{BASE}/proctor/log', headers=sh, json={'exam_id':exam_id,'violation_type':vtype})
print(f'  {len(violations)} violations logged')

print()
print('='*50)
print('✅ Full test dataset ready!')
print(f'  Exam ID: {exam_id}')
print(f'  4 students submitted, 3 different score levels')
print(f'  7 proctoring violations across 2 students')
print('='*50)


In [ ]:
# ── CELL 6: Test Module 5 report endpoints ──────────────────────────────
import requests

BASE = 'http://localhost:5000'
r = requests.post(f'{BASE}/auth/login', json={'email':'teacher@test.com','password':'teach123'})
th = {'Authorization':f'Bearer {r.json()["access_token"]}'}

print('[1] Analytics JSON:')
r = requests.get(f'{BASE}/report/analytics/1', headers=th)
d = r.json()
print(f'  Total attempts:  {d["total_attempts"]}')
print(f'  Average score:   {d["avg_score"]}/{d["exam"]["total_marks"]}  ({d["avg_percentage"]}%)')
print(f'  Pass rate:       {d["pass_rate"]}%')
print(f'  Highest:         {d["highest"]}')
print(f'  Lowest:          {d["lowest"]}')
print(f'  Score distribution: {d["score_distribution"]}')
print(f'  Violation stats: {d["violation_stats"]}')
print(f'  Questions analyzed: {len(d["question_stats"])}')
for q in d['question_stats']:
    print(f'    Q{q["question_id"]}: {q["correct_pct"]}% correct')

print()
print('[2] Downloading full class PDF...')
r = requests.get(f'{BASE}/report/exam/1.pdf', headers=th)
with open('exam_report.pdf', 'wb') as f: f.write(r.content)
print(f'  ✅ Saved: exam_report.pdf ({len(r.content)} bytes)')

print()
print('[3] Downloading individual student PDF (Danish - has violations)...')
r = requests.get(f'{BASE}/report/student/1/5.pdf', headers=th)
with open('student_report.pdf', 'wb') as f: f.write(r.content)
print(f'  ✅ Saved: student_report.pdf ({len(r.content)} bytes)')

print()
print('[4] Downloading CSV...')
r = requests.get(f'{BASE}/report/exam/1.csv', headers=th)
with open('exam_results.csv', 'wb') as f: f.write(r.content)
print(f'  ✅ Saved: exam_results.csv ({len(r.content)} bytes)')
print()
print(r.content.decode()[:500])

print()
print('✅ All Module 5 endpoints working!')
print('  Download the PDFs and CSV from the file browser on the left ←')


In [ ]:
# ── CELL 7: View database ──────────────────────────────────────────────
import sqlite3, pandas as pd

conn = sqlite3.connect('examguard.db')
print('👥 Users:')
print(pd.read_sql_query('SELECT user_id,name,email,role FROM users', conn).to_string(index=False))

print('\n📋 Exams:')
print(pd.read_sql_query('SELECT exam_id,title,duration,total_marks,is_active FROM exams', conn).to_string(index=False))

print('\n🏆 Results:')
r = pd.read_sql_query('SELECT u.name, r.score, r.total_marks, ROUND(r.score*100.0/r.total_marks,1) as pct FROM results r JOIN users u ON r.user_id=u.user_id ORDER BY r.score DESC', conn)
print(r.to_string(index=False) if len(r) else '  (no results)')

print('\n🚨 Violations by student:')
v = pd.read_sql_query('SELECT u.name, p.violation_type, COUNT(*) as count FROM proctoring_logs p JOIN users u ON p.user_id=u.user_id GROUP BY u.user_id, p.violation_type', conn)
print(v.to_string(index=False) if len(v) else '  (no violations)')
conn.close()


---
## ✅ All 5 Modules Complete!

### Module 5 adds
| Endpoint | Purpose |
|----------|---------|
| `GET /report/analytics/<exam_id>` | JSON: charts data (score distribution, pass rate, question stats, violations) |
| `GET /report/exam/<exam_id>.pdf` | Full class PDF report with charts and question analysis |
| `GET /report/exam/<exam_id>.csv` | CSV export of results |
| `GET /report/student/<exam_id>/<user_id>.pdf` | Individual student PDF (score + violations timeline) |
| `GET /my/results` | Student's own historical results |

### New frontend pages
- **`analytics.html`** — Teacher's analytics page with Chart.js charts (score distribution, pass/fail donut, question difficulty)
- **`my_results.html`** — Student's personal results history with PDF download links

### To use it
1. Run cells → copy URL from Cell 4
2. Login as **teacher@test.com** → My Exams → **📊 Analytics** button
3. See charts, then click **⬇️ Download PDF** or **⬇️ Download CSV**
4. Login as **alina@test.com** → click **My Results** → download personal PDF
